# TASK-20 · ChatOllama: parameters, streaming and experiments

**Run all** to install Ollama and both models in this Colab runtime. Expected outputs are examples; natural-language answers may differ.

## Setup: hosted accelerator and dependencies

The next cells inspect Colab's optional GPU, install **Python packages**, install required `zstd` for extraction, then install the official **Ollama Linux server**, start its API, and download two small models. On each notebook's fresh runtime the setup repeats. Official installation script executes remote code; only use the official Ollama domain.

In [ ]:
import shutil
import subprocess

print("NVIDIA GPU utility available:", bool(shutil.which("nvidia-smi")))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU assigned — expect CPU inference.")

In [ ]:
%pip install -q 'langchain-ollama>=1.0,<2' 'langchain-core>=1.0,<2' 'ollama>=0.6,<1'

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

def ollama_ready():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_ready():
    server_log = open("/tmp/ollama-server.log", "a")
    server = subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if ollama_ready():
            break
        time.sleep(2)
assert ollama_ready(), "Ollama failed to start; inspect /tmp/ollama-server.log"
print("Ollama API responding on localhost:11434")

In [ ]:
import subprocess
for model in ("qwen3:1.7b", "embeddinggemma-2:270m"):
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"], check=True)

## Generation parameters

`temperature` controls sampling diversity, `top_p` and `top_k` limit candidates, `num_ctx` sets token context length, and `num_predict` caps output tokens. `keep_alive` keeps the model loaded. `reasoning=False` simplifies the learning output.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama

settings = dict(
    model="qwen3:1.7b", base_url="http://localhost:11434",
    temperature=0.3, top_p=0.9, top_k=40,
    num_ctx=4096, num_predict=128, repeat_penalty=1.1,
    seed=42, keep_alive="10m", reasoning=False,
    validate_model_on_init=True,
)
llm = ChatOllama(**settings)
messages = [
    SystemMessage(content="You are a friendly technical tutor. Be concise."),
    HumanMessage(content="Explain RAG in three simple sentences."),
]
reply = llm.invoke(messages)
print(reply.content)
print("Usage:", reply.usage_metadata)
print("Metadata:", reply.response_metadata)

## Streaming

`stream()` yields chunks as they are generated; a full `invoke()` waits until completion.

In [ ]:
for chunk in llm.stream("Explain an embedding in one sentence."):
    print(chunk.content, end="", flush=True)
print()

## One-variable experiment

Only temperature changes. The same seed is used, but results do **not** have to differ.

In [ ]:
prompt = "Give a short analogy for retrieval-augmented generation."
for temperature in (0.1, 0.7):
    test_llm = ChatOllama(**{**settings, "temperature": temperature})
    result = test_llm.invoke(prompt)
    print(f"temperature={temperature}: {result.content}")

## Check the active model

Expected: `qwen3:1.7b` may appear in `ollama ps` while loaded; `PROCESSOR` tells you whether it uses GPU or CPU.

In [ ]:
subprocess.run(["ollama", "ps"], check=True)